# 3. How price forms

Connect market structure to price. This notebook explains marginal
pricing in the NEM, builds the **net load** concept (demand minus
renewables), and shows the "hockey stick" relationship between net
load and price that drives everything in later notebooks.

## Objectives

- Load unit-level SCADA generation data from NEMOSIS.
- Map generator locations by fuel type on the NEM map.
- Aggregate generation by fuel type and show the changing fuel mix.
- Compute net load and demonstrate its correlation with price.
- Build an empirical supply stack and overlay actual prices.
- Decompose a spike day: what happened hour by hour.
- Implement `features.net_load()` in `src/grian/features.py`.

## Prerequisites

- Notebooks 01–02 completed (parquet datasets cached).
- Internet access for NEMOSIS SCADA download (cached after first run).

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from grian.config import load_config, repo_root
from grian.data import load_prices
from grian.features import net_load
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()
warnings.filterwarnings("ignore", category=FutureWarning)

REGION = cfg["region"]
START = cfg["train_start"]
END = cfg["test_end"]
spike_threshold = cfg["spike_threshold_aud"]

REGION_COLORS = {
    "NSW1": "#2196F3", "QLD1": "#FF9800", "VIC1": "#4CAF50",
    "SA1": "#F44336", "TAS1": "#9C27B0",
}

FUEL_COLORS = {
    "Black Coal": "#333333", "Brown Coal": "#8B4513",
    "Natural Gas": "#FF6B35", "Gas": "#FF6B35",
    "Hydro": "#1E90FF", "Wind": "#2ECC71",
    "Solar": "#FFD700", "Battery": "#9B59B6",
    "Other": "#95A5A6",
}

---
## 1. Load unit SCADA data

NEMOSIS provides 5-minute generation data per unit (DUID) via the
`DISPATCH_UNIT_SCADA` table. Each row has a unit identifier, a
timestamp, and a megawatt output (`SCADAVALUE`).

We also need the **NEM Registration and Exemption List** to map each
DUID to a fuel type, region, and (sometimes) lat/lon. NEMOSIS provides
this as a static table.

In [ ]:
from nemosis import dynamic_data_compiler, static_table

cache = str(Path(cfg["nemosis_cache"]))
Path(cache).mkdir(parents=True, exist_ok=True)

# Load the registration list for fuel-type mapping.
# The AEMO endpoint sometimes returns an error page instead of the
# Excel file, so we fall back to a hardcoded SA1 DUID mapping.
try:
    rego = static_table("Generators and Scheduled Loads", cache)
    print(f"Loaded registration list: {len(rego)} rows, {rego['DUID'].nunique()} DUIDs")
except Exception as e:
    rego = None
    print(f"Could not load registration list ({type(e).__name__}: {e})")
    print("Using hardcoded SA1 DUID-to-fuel mapping instead.")

In [ ]:
# Simplified fuel categories used throughout this notebook
fuel_map = {
    "Black Coal": "Black Coal", "Brown Coal": "Brown Coal",
    "Natural Gas / Fuel Oil": "Gas", "Natural Gas": "Gas",
    "Water": "Hydro", "Wind": "Wind", "Solar": "Solar",
    "Battery": "Battery", "Diesel": "Gas",
}

if rego is not None:
    if "Fuel Source - Primary" in rego.columns:
        fuel_col = "Fuel Source - Primary"
    elif "FuelSourcePrimary" in rego.columns:
        fuel_col = "FuelSourcePrimary"
    else:
        fuel_col = [c for c in rego.columns if "fuel" in c.lower()][0]
    print(f"Fuel column: {fuel_col}")
    print(rego[fuel_col].value_counts())
else:
    fuel_col = None
    print("No registration list available — will use hardcoded mapping.")

In [ ]:
# Build DUID-to-fuel mapping
if rego is not None:
    if "Region" in rego.columns:
        region_col = "Region"
    else:
        region_col = [c for c in rego.columns if "region" in c.lower()][0]
    duid_fuel = rego[["DUID", fuel_col, region_col]].drop_duplicates(subset="DUID")
    duid_fuel = duid_fuel.rename(columns={fuel_col: "fuel", region_col: "region"})
    duid_fuel["fuel_simple"] = duid_fuel["fuel"].map(fuel_map).fillna("Other")
else:
    # Hardcoded SA1 DUID-to-fuel mapping (major scheduled/semi-scheduled units).
    # Source: AEMO NEM Registration and Exemption List (cached snapshot).
    _sa1_duids = {
        # Gas
        "TORRB1": "Gas", "TORRB2": "Gas", "TORRB3": "Gas", "TORRB4": "Gas",
        "TORRA1": "Gas", "TORRA2": "Gas", "TORRA3": "Gas", "TORRA4": "Gas",
        "OSBORNE1": "Gas", "PELICAN1": "Gas", "PELICAN2": "Gas",
        "PPCCGT": "Gas", "LONSDALE": "Gas", "DRYCGT1": "Gas",
        "LADBROK1": "Gas", "LADBROK2": "Gas", "MINTARO": "Gas",
        "SNUG1": "Gas", "ANGAST1": "Gas", "BARMDl1": "Gas",
        "BARKIPS1": "Gas", "APTS1": "Gas",
        # Wind
        "HDWF1": "Wind", "HDWF2": "Wind", "HDWF3": "Wind",
        "LKBONNY1": "Wind", "LKBONNY2": "Wind", "LKBONNY3": "Wind",
        "SNOWNTH1": "Wind", "SNOWSTH1": "Wind",
        "WATERLWF": "Wind", "CATHROCK": "Wind", "CLEMGPWF": "Wind",
        "HALLWF1": "Wind", "HALLWF2": "Wind",
        "NBHWF1": "Wind", "LGAPWF1": "Wind",
        "WGWF1": "Wind", "WPWF": "Wind", "MUWAWF1": "Wind",
        "STARHLWF": "Wind", "KEYNWF1": "Wind",
        "PORTWF": "Wind", "YANDINWF": "Wind",
        "CNUNDAWF": "Wind", "WKWF1": "Wind", "MLWF1": "Wind",
        "LCWF1": "Wind", "CROOKWF1": "Wind", "RYANLWF1": "Wind",
        "BROCKNWF": "Wind", "GDNWF1": "Wind",
        "BWDWF1": "Wind", "MTMWF1": "Wind",
        "PALMWF1": "Wind", "PALMWF2": "Wind",
        "BONSHAW1": "Wind", "CSPVPS1": "Wind",
        "LINCOLN1": "Wind",
        # Solar
        "BDALEX": "Solar", "BDAL1": "Solar",
        "TAILEM1": "Solar", "TSBSF1": "Solar",
        "BNGSF1": "Solar", "BNGSF2": "Solar",
        "SCSF1": "Solar", "WHYALLASF": "Solar",
        "MKSF1": "Solar", "LBBSF1": "Solar",
        "MOREE1": "Solar", "MOREE2": "Solar",
        # Battery
        "HPRG1": "Battery", "DALNTHL1": "Battery",
        "HPRL1": "Battery", "BDALBL1": "Battery",
        "LBBBL1": "Battery",
    }
    rows = [{"DUID": d, "fuel_simple": f, "region": "SA1"}
            for d, f in _sa1_duids.items()]
    duid_fuel = pd.DataFrame(rows)

sa1_duids = duid_fuel[duid_fuel["region"] == REGION]
print(f"\n{REGION} generators by fuel:")
print(sa1_duids["fuel_simple"].value_counts())

---
## 2. Map: generator locations by fuel type

If latitude/longitude are available in the registration list, we can
map where each generator sits. This gives geographic intuition for
why wind is concentrated in some areas and gas peakers in others.

In [ ]:
# Generator map — only possible when the full registration list loaded
regions_gdf = gpd.read_file(repo_root() / "data" / "nem_regions.geojson")

if rego is not None:
    lat_col = [c for c in rego.columns if "lat" in c.lower()]
    lon_col = [c for c in rego.columns if "lon" in c.lower()]
else:
    lat_col, lon_col = [], []

if lat_col and lon_col:
    gen_locs = rego[["DUID", fuel_col, lat_col[0], lon_col[0]]].dropna()
    gen_locs = gen_locs.rename(columns={lat_col[0]: "lat", lon_col[0]: "lon"})
    gen_locs["fuel_simple"] = gen_locs[fuel_col].map(fuel_map).fillna("Other")

    fig, ax = plt.subplots(figsize=(10, 12))
    regions_gdf.plot(ax=ax, color="#f0f0f0", edgecolor="white", linewidth=1.5)

    for fuel, color in FUEL_COLORS.items():
        subset = gen_locs[gen_locs["fuel_simple"] == fuel]
        if len(subset) > 0:
            ax.scatter(subset["lon"], subset["lat"], c=color,
                       s=20, label=fuel, alpha=0.7, edgecolor="none")

    ax.set_xlim(112, 155)
    ax.set_ylim(-45, -10)
    ax.legend(loc="upper left", fontsize=9)
    ax.set_title("NEM generators by fuel type", fontsize=14)
    save_fig(fig, "03_generator_map")
    plt.show()
else:
    print("No lat/lon data available (registration list not loaded).")
    print("Skipping generator map — fuel mix charts below still work.")

---
## 3. Fuel mix: generation by fuel type over time

Load the SCADA data for a representative sample period and aggregate
by fuel type. We use a shorter window here to keep download times
manageable — the full 4.5-year SCADA dataset is very large.

In [ ]:
# Use a 3-month sample for the fuel-mix analysis
sample_start = "2024-01-01"
sample_end = "2024-03-31"

start_fmt = pd.Timestamp(sample_start).strftime("%Y/%m/%d %H:%M:%S")
end_fmt = pd.Timestamp(sample_end).strftime("%Y/%m/%d %H:%M:%S")

scada = dynamic_data_compiler(
    start_fmt, end_fmt, "DISPATCH_UNIT_SCADA", cache,
    select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
)

scada = scada.rename(columns={"SETTLEMENTDATE": "timestamp", "SCADAVALUE": "mw"})
scada["timestamp"] = pd.to_datetime(scada["timestamp"])
# Shift to interval-start
scada["timestamp"] = scada["timestamp"] - pd.Timedelta(minutes=5)

print(f"SCADA rows: {len(scada):,}")
print(f"Unique DUIDs: {scada['DUID'].nunique()}")
scada.head()

In [ ]:
# Join fuel type and filter to SA1
scada = scada.merge(duid_fuel[["DUID", "fuel_simple", "region"]], on="DUID", how="left")
scada_sa1 = scada[scada["region"] == REGION].copy()

# Aggregate by timestamp and fuel type
fuel_gen = scada_sa1.groupby(["timestamp", "fuel_simple"])["mw"].sum().unstack(fill_value=0)
# Resample to hourly for a cleaner plot
fuel_hourly = fuel_gen.resample("1h").mean()

print(f"SA1 fuel types present: {list(fuel_hourly.columns)}")
fuel_hourly.head()

In [ ]:
# Stacked area chart of SA1 generation by fuel
plot_cols = [c for c in ["Gas", "Wind", "Solar", "Battery", "Other"]
             if c in fuel_hourly.columns]
plot_colors = [FUEL_COLORS.get(c, "#95A5A6") for c in plot_cols]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(fuel_hourly.index, [fuel_hourly[c].values for c in plot_cols],
             labels=plot_cols, colors=plot_colors, alpha=0.8)
ax.set_ylabel("Generation (MW)")
ax.set_title(f"{REGION} — generation by fuel type ({sample_start} to {sample_end})")
ax.legend(loc="upper right")
fig.tight_layout()
save_fig(fig, "03_fuel_mix_stacked")
plt.show()

SA1's fuel mix is dominated by wind and solar, with gas peakers
filling the gaps. The daily solar cycle is clearly visible — and
when solar drops off in the evening, gas must ramp up quickly.

---
## 4. Net load and the hockey stick

**Net load** = demand − wind − solar. It represents the residual
demand that must be met by dispatchable generators (gas, hydro,
batteries, imports). When net load is high, expensive peakers run
and prices spike. When net load is low or negative, renewables
flood the market and prices collapse.

This is implemented in `grian.features.net_load()`.

In [ ]:
# Compute net load from SCADA data
wind_gen = fuel_gen["Wind"] if "Wind" in fuel_gen.columns else pd.Series(0, index=fuel_gen.index)
solar_gen = fuel_gen["Solar"] if "Solar" in fuel_gen.columns else pd.Series(0, index=fuel_gen.index)

# Load demand for the sample period
from grian.data import load_demand

demand_sample = load_demand(REGION, sample_start, sample_end, cache=cfg["nemosis_cache"])

# Align on common index
common = pd.DataFrame({
    "demand": demand_sample["demand"],
    "wind": wind_gen,
    "solar": solar_gen,
}).dropna()

common["net_load"] = net_load(common["demand"], common["solar"], common["wind"])

print(f"Net load range: {common['net_load'].min():.0f} to {common['net_load'].max():.0f} MW")
print(f"Negative net load (renewables > demand): {(common['net_load'] < 0).mean():.1%}")

In [ ]:
# Load prices for the same period and join
prices_sample = load_prices(REGION, sample_start, sample_end, cache=cfg["nemosis_cache"])
scatter_df = common.join(prices_sample, how="inner")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Net load vs price scatter — the hockey stick
ax1.scatter(scatter_df["net_load"], scatter_df["price"],
            s=1, alpha=0.15, c="C0")
ax1.set_xlabel("Net load (MW)")
ax1.set_ylabel("Price ($/MWh)")
ax1.set_title("Net load vs price — the hockey stick")
ax1.set_yscale("symlog", linthresh=100)
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax1.axvline(0, color="gray", linewidth=0.5, linestyle="--")

# Binned median — cleaner view of the relationship
scatter_df["nl_bin"] = pd.cut(scatter_df["net_load"], bins=50)
binned = scatter_df.groupby("nl_bin", observed=True)["price"].agg(["median", "mean", "count"])
binned["midpoint"] = [iv.mid for iv in binned.index]
ax2.plot(binned["midpoint"], binned["median"], "o-", markersize=3, label="Median")
ax2.plot(binned["midpoint"], binned["mean"], "s-", markersize=3, label="Mean", alpha=0.7)
ax2.set_xlabel("Net load (MW)")
ax2.set_ylabel("Price ($/MWh)")
ax2.set_title("Binned net load vs price")
ax2.legend()
ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")

fig.tight_layout()
save_fig(fig, "03_net_load_vs_price")
plt.show()

The "hockey stick": price is relatively flat when net load is moderate
(plenty of cheap capacity available), then shoots up exponentially
when net load pushes into the range where expensive gas peakers must
run. At negative net load, prices collapse or go negative.

This is the **merit-order effect** of renewables: wind and solar push
expensive generators off the stack, lowering the clearing price. When
they disappear (calm night), only expensive generators remain.

---
## 5. Empirical supply stack

The NEM clears by stacking generator offers from cheapest to most
expensive. We can approximate this by ranking fuel types by typical
marginal cost and building a step function.

In [ ]:
# Approximate marginal costs by fuel type (indicative, $/MWh)
marginal_costs = {
    "Solar": 0, "Wind": 0, "Hydro": 15,
    "Brown Coal": 25, "Black Coal": 40,
    "Gas": 80, "Battery": 100, "Other": 120,
}

# Average capacity by fuel type in SA1 during the sample
avg_capacity = fuel_gen.mean().sort_values(ascending=False)
stack_fuels = [f for f in marginal_costs.keys() if f in avg_capacity.index]

# Build the step function
cumulative_mw = 0
stack_x, stack_y = [0], [0]
for fuel in sorted(stack_fuels, key=lambda f: marginal_costs[f]):
    cap = avg_capacity.get(fuel, 0)
    if cap > 0:
        stack_x.extend([cumulative_mw, cumulative_mw + cap])
        stack_y.extend([marginal_costs[fuel], marginal_costs[fuel]])
        cumulative_mw += cap

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(stack_x, stack_y, "k-", linewidth=2, label="Approx supply stack")

# Color-fill each fuel block
cumulative_mw = 0
for fuel in sorted(stack_fuels, key=lambda f: marginal_costs[f]):
    cap = avg_capacity.get(fuel, 0)
    if cap > 0:
        ax.fill_between([cumulative_mw, cumulative_mw + cap],
                        0, marginal_costs[fuel],
                        alpha=0.3, color=FUEL_COLORS.get(fuel, "gray"),
                        label=f"{fuel} ({cap:.0f} MW avg)")
        cumulative_mw += cap

# Overlay median price as a horizontal line
median_price = prices_sample["price"].median()
ax.axhline(median_price, color="red", linewidth=1.5, linestyle="--",
           label=f"Median price ${median_price:.0f}")

ax.set_xlabel("Cumulative generation (MW)")
ax.set_ylabel("Marginal cost ($/MWh)")
ax.set_title(f"{REGION} — approximate supply stack")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
save_fig(fig, "03_supply_stack")
plt.show()

The stack shows why net load matters: as demand increases, the market
climbs the stack into more expensive fuel sources. The steep right
end (gas peakers) is where prices become very sensitive to small
changes in demand — a 100 MW increase can mean a \$200/MWh price jump.

---
## 6. Spike decomposition — anatomy of a spike day

Pick a day with a significant price spike and decompose what happened
to generation, demand, and net load hour by hour.

In [ ]:
# Find the day with the highest max price in our sample
daily_max = prices_sample["price"].resample("D").max()
spike_day = daily_max.idxmax().strftime("%Y-%m-%d")
print(f"Spike day: {spike_day} (max price: ${daily_max.max():,.0f}/MWh)")

# Data for that day
day_prices = prices_sample.loc[spike_day]
day_gen = fuel_gen.loc[spike_day] if spike_day in fuel_gen.index else fuel_gen.iloc[:0]
day_demand = demand_sample.loc[spike_day] if spike_day in demand_sample.index.strftime("%Y-%m-%d") else demand_sample.iloc[:0]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Price
axes[0].plot(day_prices.index, day_prices["price"], "k-", linewidth=1.5)
axes[0].axhline(spike_threshold, color="red", linewidth=0.8, linestyle="--",
                label=f"${spike_threshold} threshold")
axes[0].set_ylabel("Price ($/MWh)")
axes[0].set_title(f"{REGION} — spike day: {spike_day}")
axes[0].legend()

# Generation by fuel
if len(day_gen) > 0:
    for fuel in plot_cols:
        if fuel in day_gen.columns:
            axes[1].plot(day_gen.index, day_gen[fuel], linewidth=1.2,
                         color=FUEL_COLORS.get(fuel, "gray"), label=fuel)
    axes[1].set_ylabel("Generation (MW)")
    axes[1].legend(loc="upper right", fontsize=9)
    axes[1].set_title("Generation by fuel")

# Net load
if spike_day in scatter_df.index.strftime("%Y-%m-%d"):
    day_nl = scatter_df.loc[spike_day, "net_load"]
    if hasattr(day_nl, 'index'):
        axes[2].plot(day_nl.index, day_nl.values, linewidth=1.5, color="C3")
axes[2].set_ylabel("Net load (MW)")
axes[2].set_title("Net load (demand − renewables)")

fig.tight_layout()
save_fig(fig, "03_spike_day_decomposition")
plt.show()

The pattern is typically: solar drops off in the evening → net load
spikes → gas peakers ramp up but may not be fast enough or have
enough capacity → price spikes. Understanding this chain is why
net load is the single most important feature for price forecasting.

---
## 7. Which fuel is marginal at different net-load levels?

The marginal (price-setting) fuel changes as net load moves. At low
net load, renewables are marginal (price near zero). At high net load,
gas is marginal (price \$80+).

In [ ]:
# For each net-load quintile, show the fuel mix
scatter_with_gen = scatter_df.join(fuel_gen, how="inner", rsuffix="_gen")
scatter_with_gen["nl_quintile"] = pd.qcut(scatter_with_gen["net_load"], q=5,
                                           labels=["Very Low", "Low", "Mid", "High", "Very High"])

gen_cols = [c for c in plot_cols if c in scatter_with_gen.columns]
quintile_fuel = scatter_with_gen.groupby("nl_quintile", observed=True)[gen_cols].mean()

fig, ax = plt.subplots(figsize=(10, 6))
quintile_fuel.plot(kind="bar", stacked=True, ax=ax,
                    color=[FUEL_COLORS.get(c, "gray") for c in gen_cols],
                    alpha=0.8)
ax.set_ylabel("Average generation (MW)")
ax.set_xlabel("Net load quintile")
ax.set_title(f"{REGION} — fuel mix by net-load level")
ax.legend(loc="upper left")
plt.xticks(rotation=0)
fig.tight_layout()
save_fig(fig, "03_fuel_by_net_load")
plt.show()

# Median price per quintile
print("\nMedian price by net-load quintile:")
for q in ["Very Low", "Low", "Mid", "High", "Very High"]:
    subset = scatter_with_gen[scatter_with_gen["nl_quintile"] == q]
    if len(subset) > 0:
        print(f"  {q:>10}: ${subset['price'].median():.1f}/MWh")

---
## Exercises

### Exercise 1: At what net-load level does gas become marginal?

Find the net-load threshold where gas generation starts ramping
significantly and prices begin to rise above \$50/MWh.

<details><summary>Hint 1</summary>

Bin net load into 20 bins with `pd.cut`, then compute the mean gas
generation and mean price per bin. Plot both on twin y-axes.

</details>

<details><summary>Hint 2</summary>

The threshold is where gas generation ramps from near-zero to a
significant fraction of total supply. Look for the "elbow" in the
gas-vs-net-load curve.

</details>

<details><summary>Hint 3</summary>

Compare the gas ramp-up threshold with the price inflection point.
They should be close — gas becoming marginal is what drives the
hockey-stick bend.

</details>

<details><summary>Solution</summary>

```python
df_ex = scatter_with_gen.copy()
df_ex["nl_bin"] = pd.cut(df_ex["net_load"], bins=20)
binned_ex = df_ex.groupby("nl_bin", observed=True).agg(
    price_median=("price", "median"),
    gas_mean=("Gas", "mean") if "Gas" in df_ex.columns else ("net_load", "count"),
    midpoint=("net_load", "median"),
)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

ax1.bar(range(len(binned_ex)), binned_ex["gas_mean"], alpha=0.4,
        color=FUEL_COLORS["Gas"], label="Gas generation (MW)")
ax2.plot(range(len(binned_ex)), binned_ex["price_median"], "ko-",
         markersize=4, label="Median price")

ax1.set_xlabel("Net load bin")
ax1.set_ylabel("Gas generation (MW)", color=FUEL_COLORS["Gas"])
ax2.set_ylabel("Median price ($/MWh)")
ax1.set_title(f"{REGION} — gas ramp-up vs price")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")

fig.tight_layout()
plt.show()

# Find the approximate threshold
if "Gas" in df_ex.columns:
    gas_active = df_ex[df_ex["Gas"] > 50]
    print(f"Gas gen > 50 MW starts at net load ~{gas_active['net_load'].quantile(0.1):.0f} MW")
    print(f"Median price when gas > 50 MW: ${gas_active['price'].median():.0f}/MWh")
```

Gas generation ramps up sharply once net load exceeds a threshold
that depends on the available renewable capacity. This is the point
where renewables can no longer meet demand alone, and the clearing
price jumps from near-zero to gas marginal cost (\$60–120/MWh).
This threshold shifts left over time as more renewables are installed.

</details>

In [ ]:
# Your analysis here

### Exercise 2: How does the hockey stick change over time?

Compare the net-load vs price relationship between 2020–2021 and
2023–2024. Has the curve shifted as more solar was installed?

<details><summary>Hint 1</summary>

You'll need SCADA data for both periods. To keep things manageable,
use a single representative month from each period (e.g. January 2021
vs January 2024).

</details>

<details><summary>Hint 2</summary>

Plot the binned net-load vs price for each period on the same axes.
If more solar has been installed, the curve should shift left —
lower net load at the same demand level.

</details>

<details><summary>Hint 3</summary>

The shift represents the merit-order effect of new renewable
capacity. Each new MW of solar displaces gas at the margin,
lowering the net-load level at which prices start to rise.

</details>

<details><summary>Solution</summary>

```python
# This exercise requires loading SCADA for an additional period.
# We compare Jan 2021 vs Jan 2024 as representative months.

periods = {
    "Jan 2021": ("2021-01-01", "2021-01-31"),
    "Jan 2024": ("2024-01-01", "2024-01-31"),
}

fig, ax = plt.subplots(figsize=(12, 6))

for label, (s, e) in periods.items():
    s_fmt = pd.Timestamp(s).strftime("%Y/%m/%d %H:%M:%S")
    e_fmt = pd.Timestamp(e).strftime("%Y/%m/%d %H:%M:%S")

    sc = dynamic_data_compiler(s_fmt, e_fmt, "DISPATCH_UNIT_SCADA", cache,
                               select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"])
    sc = sc.rename(columns={"SETTLEMENTDATE": "timestamp", "SCADAVALUE": "mw"})
    sc["timestamp"] = pd.to_datetime(sc["timestamp"]) - pd.Timedelta(minutes=5)
    sc = sc.merge(duid_fuel[["DUID", "fuel_simple", "region"]], on="DUID", how="left")
    sc = sc[sc["region"] == REGION]

    fg = sc.groupby(["timestamp", "fuel_simple"])["mw"].sum().unstack(fill_value=0)
    w = fg["Wind"] if "Wind" in fg.columns else 0
    sol = fg["Solar"] if "Solar" in fg.columns else 0

    dem = load_demand(REGION, s, e, cache=cfg["nemosis_cache"])
    pr = load_prices(REGION, s, e, cache=cfg["nemosis_cache"])

    tmp = pd.DataFrame({"demand": dem["demand"], "wind": w, "solar": sol}).dropna()
    tmp["net_load"] = net_load(tmp["demand"], tmp["solar"], tmp["wind"])
    tmp = tmp.join(pr, how="inner")

    tmp["nl_bin"] = pd.cut(tmp["net_load"], bins=30)
    binned = tmp.groupby("nl_bin", observed=True).agg(
        price_med=("price", "median"), nl_mid=("net_load", "median")
    )
    ax.plot(binned["nl_mid"], binned["price_med"], "o-", markersize=3, label=label)

ax.set_xlabel("Net load (MW)")
ax.set_ylabel("Median price ($/MWh)")
ax.set_title(f"{REGION} — hockey stick: 2021 vs 2024")
ax.legend()
fig.tight_layout()
plt.show()
```

More solar installed by 2024 shifts the curve left — the same demand
level produces lower net load, so prices are lower at the same demand.
The shape of the hockey stick remains, but the inflection point moves.
This structural change means a model trained on 2021 data would
systematically overpredict prices in 2024 at moderate demand levels.

</details>

In [ ]:
# Your analysis here

### Exercise 3: Net-load → price regression

Fit a simple regression from net load to price. How much of the
variance does it explain? Is the relationship linear?

<details><summary>Hint 1</summary>

Use `sklearn.linear_model.LinearRegression` for the linear fit.
Report R² on the full sample. Then try a polynomial (degree 2 or 3)
via `sklearn.preprocessing.PolynomialFeatures`.

</details>

<details><summary>Hint 2</summary>

The linear R² will be modest because the relationship is nonlinear.
Try regressing on `arcsinh(price)` instead of raw price — the
transformed target may be more linear in net load.

</details>

<details><summary>Hint 3</summary>

Plot residuals vs net load. If the relationship were truly linear,
residuals would be homoscedastic. The fan shape you'll see confirms
nonlinearity and heteroscedasticity — motivating the ML models in
later notebooks.

</details>

<details><summary>Solution</summary>

```python
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

X = scatter_df[["net_load"]].values
y_raw = scatter_df["price"].values
y_asinh = np.arcsinh(y_raw)

# Linear on raw price
lr = LinearRegression().fit(X, y_raw)
print(f"Linear on raw price:     R² = {lr.score(X, y_raw):.3f}")

# Linear on arcsinh price
lr_t = LinearRegression().fit(X, y_asinh)
print(f"Linear on arcsinh price: R² = {lr_t.score(X, y_asinh):.3f}")

# Polynomial degree 3 on raw price
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)
lr_p = LinearRegression().fit(X_poly, y_raw)
print(f"Poly(3) on raw price:    R² = {lr_p.score(X_poly, y_raw):.3f}")

# Residual plot
resid = y_raw - lr.predict(X)
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(scatter_df["net_load"], resid, s=1, alpha=0.1)
ax.axhline(0, color="red", linewidth=0.8)
ax.set_xlabel("Net load (MW)")
ax.set_ylabel("Residual ($/MWh)")
ax.set_title("Linear regression residuals — clearly nonlinear")
ax.set_yscale("symlog", linthresh=100)
fig.tight_layout()
plt.show()
```

The linear R² is low — net load alone explains only a modest fraction
of price variance. The polynomial improves things but the residuals
still show massive heteroscedasticity at high net load. This confirms
that (a) we need more features beyond net load, and (b) nonlinear
models will outperform linear ones, especially in the tails.

</details>

In [ ]:
# Your analysis here

---
## What we learned

1. The NEM clears by dispatching generators in merit order — cheapest
   first. The marginal (last dispatched) generator sets the price.
2. **Net load** (demand − wind − solar) is the key driver: it tells
   you where on the supply stack the market is clearing.
3. The net-load vs price relationship is a **hockey stick** — flat
   when renewables dominate, exponential when gas peakers must run.
4. SA1's fuel mix (wind + solar + gas) creates extreme price dynamics:
   negative prices at midday, spikes in the evening.
5. The hockey stick shifts over time as more renewables are installed,
   so models need features that capture the *current* supply mix.
6. Net load alone explains only a fraction of price variance — we
   need calendar, lag, and weather features too.

**Next:** Notebook 04 adds weather data (ERA5) and generation forecasts
to build the renewable features that drive net load.

In [ ]:
# Write report
report_dir = Path(cfg["paths"]["reports"])
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 03 — Price Formation Report

Region: {REGION} | Sample: {sample_start} to {sample_end}

## Key findings

- Net load range: {common['net_load'].min():.0f} to {common['net_load'].max():.0f} MW
- Negative net load (renewables > demand): {(common['net_load'] < 0).mean():.1%}
- SA1 fuel mix: primarily wind, solar, and gas peakers
- Hockey-stick relationship confirmed between net load and price
- Gas becomes marginal when net load exceeds ~{common['net_load'].quantile(0.7):.0f} MW

## Implemented

- `features.net_load()` — demand minus wind and solar generation
"""

(report_dir / "03_price_formation.md").write_text(report)
print("Report written to", report_dir / "03_price_formation.md")